In [2]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cu130
True


In [2]:
import os
import cv2
import numpy as np
import yaml
import shutil
from tqdm import tqdm
from ultralytics import YOLO

In [5]:
SRC_IMG_TRAIN = "C:/training/segmentation_clean/Images/ann_dir/train"
SRC_IMG_VAL   = "C:/training/segmentation_clean/Images/ann_dir/test"
SRC_ANN_TRAIN = "C:/training/segmentation_clean/Images/ann_dir/train"
SRC_ANN_VAL   = "C:/training/segmentation_clean/Images/ann_dir/test"

CATEGORY_FILE = "C:/training/segmentation_clean/category_id.txt"
YOLO_ROOT = "yolo_dataset_clean"


In [6]:
dirs = [
    f"{YOLO_ROOT}/images/train", f"{YOLO_ROOT}/images/val",
    f"{YOLO_ROOT}/labels/train", f"{YOLO_ROOT}/labels/val"
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

In [7]:
classes = []
with open(CATEGORY_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            parts = line.strip().split()
            name = parts[-1] 
            classes.append(name)
print(f"Загружено классов: {len(classes)}")

Загружено классов: 104


In [8]:
def convert_and_copy(img_src, ann_src, img_dest, label_dest):
    print(f"Обработка: {img_src}...")
    
    mask_files = [f for f in os.listdir(ann_src) if f.endswith('.png')]
    
    for mask_name in tqdm(mask_files):
        img_name = mask_name.replace('.png', '.jpg')
        src_img_path = os.path.join(img_src, img_name)
        
        
        if not os.path.exists(src_img_path):
            img_name = mask_name.replace('.png', '.png')
            src_img_path = os.path.join(img_src, img_name)

        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, os.path.join(img_dest, img_name))
        else:
            continue

        mask_path = os.path.join(ann_src, mask_name)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None: continue
        
        h, w = mask.shape[:2]
        cls_ids = np.unique(mask)
        
        txt_name = mask_name.rsplit('.', 1)[0] + ".txt"
        with open(os.path.join(label_dest, txt_name), 'w') as f:
            for cls_id in cls_ids:
                if cls_id == 0: continue
                
                binary_mask = np.where(mask == cls_id, 255, 0).astype(np.uint8)
                contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for contour in contours:
                    if len(contour) < 3: continue 
                    
                    polygon = contour.reshape(-1, 2) / [w, h]
                    poly_str = " ".join([f"{c[0]:.4f} {c[1]:.4f}" for c in polygon])
                    
                    f.write(f"{int(cls_id)-1} {poly_str}\n")

In [9]:
convert_and_copy(SRC_IMG_TRAIN, SRC_ANN_TRAIN, f"{YOLO_ROOT}/images/train", f"{YOLO_ROOT}/labels/train")
convert_and_copy(SRC_IMG_VAL, SRC_ANN_VAL, f"{YOLO_ROOT}/images/val", f"{YOLO_ROOT}/labels/val")

Обработка: C:/training/segmentation_clean/Images/ann_dir/train...


  9%|▉         | 461/4944 [00:09<01:33, 47.79it/s]


KeyboardInterrupt: 

In [10]:
data_yaml = {
    'path': os.path.abspath(YOLO_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'names': {i: name for i, name in enumerate(classes)}
}

with open('food103_clean.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("Данные готовы! Файл food103.yaml создан.")

Данные готовы! Файл food103.yaml создан.


In [3]:
model = YOLO('yolo26l-seg.pt') 

results = model.train(
    data='C:/training/segmentation_clean/food103_clean.yaml',
    epochs=300,
    
    imgsz=640,              
    batch=16,               
    device=0,               
    
    retina_masks=True,
    
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3.0,
    weight_decay=0.0005,
    
    box=7.5,
    cls=0.5,
    dfl=1.5,
    
    hsv_h=0.015,         
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,             
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,              
   
    save_period=10,
    patience=50,
    workers=4,           
    amp=True,               
)

New https://pypi.org/project/ultralytics/8.4.45 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.22  Python-3.14.2 torch-2.10.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:/training/segmentation_clean/food103_clean.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26l-seg.pt, momentum=0.937, mosaic=1.